In [1]:
import pdfplumber


dkt = pdfplumber.open("driver-knowledge-test-questions-car.pdf")
print("pages:", len(dkt.pages))
print(dkt.pages[5].extract_text())

pages: 94
CG037 - General Knowledge RUH
When can a private car travel in a lane marked by this sign?
- Only within 100 metres of making a turn.
- When carrying at least two passengers.
- Only to overtake another vehicle.
CG038 - General Knowledge RUH
Are you permitted to drive a car towing more than one trailer?
- No, not at all.
- Yes, if the combined length of vehicle and trailers does not exceed 15 metres.
- Yes, provided you have held a licence for two years.
CG039 - General Knowledge RUH
What must you do when you are towing a caravan to help other vehicles overtake?
- Keep at least 60 metres behind heavy vehicles or other vehicles
towing caravans.
- Drive at least 25 km/h below the speed limit.
- Stop immediately and let the faster vehicle overtake.
CG040 - General Knowledge RUH
Are you permitted to tow a caravan with a person riding in the caravan?
- No, not under any circumstances.
- Yes, provided you do not exceed 60 km/h.
- Yes, if the person(s) in the caravan are over 12 year

In [2]:
# check the last few pages
for i in [90, 91, 92, 93]:
    print(f"===== PAGE {i+1} =====")
    print(dkt.pages[i].extract_text()[:800])
    print()

===== PAGE 91 =====
SI048 – Traffic Signs RUH
What does this sign mean?
- Vehicles carrying three or more people, and
motorcycles may use this lane.
- Motor cars carrying one passenger may use this lane.
- This lane is reserved for bicycle riders only.
SI049 - Traffic Signs RUH
What does this sign mean?
- Railway level crossing ahead, slow down and
look out for trains, be prepared to stop.
- Cattle or sheep crossing the road ahead, drive slowly.
- Give way sign ahead, slow down and be prepared to stop.
SI050 – Traffic Signs RUH
When you see this sign, you should -
- Drive carefully and watch out for traffic entering
from a side-road intersection ahead.
- Take care, trucks crossing the road ahead.
- Take a break, rest area ahead.
SI051 – Traffic Signs RUH
When you see this sign you must -
- Stop at the stop li

===== PAGE 92 =====
SI052 - Traffic Signs RUH
When you see this sign you must -
- Slow down and be prepared to stop and give way to
all traffic if there is any danger of a collis

In [3]:
for i in [0, 1, 2]:
    print(f"===== PAGE {i+1} =====")
    print(dkt.pages[i].extract_text()[:800])
    print()

===== PAGE 1 =====
DRIVER KNOWLEDGE TEST QUESTIONS
CLASS C (CAR) LICENCE
ICAC1 - ICAC RUH
What will happen if you are caught cheating on the knowledge test?
- You will not be allowed to take another test for 6 weeks.
- You will never be allowed to take another test.
- Nothing, there is no penalty.
ICAC2 - ICAC RUH
What will happen if you offer a testing officer a bribe to pass your driving test?
- Action will be taken against you. The penalties are severe and
include fines and imprisonment.
- Only the testing officer will be investigated.
- Nothing, there is no penalty.
ICAC3 - ICAC RUH
What will happen if a testing officer asks you for a bribe to pass your driving test and you give it to
him or her?
- Action will be taken against all involved. The penalties are severe
and include fines and imprisonment.
- N

===== PAGE 2 =====
GENERAL KNOWLEDGE SECTION
CG001 – General Knowledge RUH
Can a P1 or P2 provisional driver legally instruct a learner driver?
- No.
- Yes, provided the provision

In [5]:
def look(term, n=2):
    hits = [p for p in pages if term.lower() in p["text"].lower()]
    for h in hits[:n]:
        print(f"--- p.{h['page']} ---")
        print(h["text"][:700], "\n")

look("P1 provisional")

NameError: name 'pages' is not defined

In [7]:
import pdfplumber, re

dkt = pdfplumber.open("driver-knowledge-test-questions-car.pdf")
full_text = "\n".join((p.extract_text() or "") for p in dkt.pages)

blocks = re.split(r"\n(?=[A-Z]{2,4}\d{1,3}\s*[-–]\s*)", full_text)

questions = []
for b in blocks:
    m = re.match(r"([A-Z]{2,4}\d{1,3})\s*[-–]\s*(.*)", b.strip())
    if not m:
        continue
    qid = m.group(1)
    section = m.group(2).strip()
    lines = b.strip().split("\n")[1:]

    q_lines, options, cur = [], [], None
    for line in lines:
        s = line.strip()
        if not s or re.fullmatch(r"\d{1,3}", s):
            continue
        if s.startswith("- "):
            if cur: options.append(cur)
            cur = s[2:].strip()
        elif cur is not None:
            cur += " " + s
        else:
            q_lines.append(s)
    if cur: options.append(cur)

    if q_lines and options:
        questions.append({
            "dkt_id": qid,
            "section": section,
            "question": " ".join(q_lines),
            "options": options,
            "answer": options[0],
        })

print(f"parsed {len(questions)}")

from collections import Counter
print(Counter(len(q["options"]) for q in questions))

parsed 357
Counter({3: 356, 1: 1})


In [8]:
bad = [q for q in questions if len(q["options"]) != 3]
print(bad)

[{'dkt_id': 'PD031', 'section': 'Pedestrians RUH', 'question': 'You must give way to pedestrians - -. At all times, if there is any danger of colliding with them, even if there is no marked crossing. -.Only on marked foot crossings.', 'options': ['Only on marked foot crossings and traffic light pedestrian crossings.'], 'answer': 'Only on marked foot crossings and traffic light pedestrian crossings.'}]


In [9]:
questions = [q for q in questions if len(q["options"]) == 3]
print(len(questions))

356


In [10]:
IMG = re.compile(r"\bthis sign\b|\bthis diagram\b|\bshown\b|\bpicture\b|\bimage\b|\billustrat|\bthis symbol\b", re.I)

image_qs = [q for q in questions if IMG.search(q["question"])]
text_qs  = [q for q in questions if not IMG.search(q["question"])]

print(f"total:           {len(questions)}")
print(f"image-dependent: {len(image_qs)}")
print(f"text-answerable: {len(text_qs)}")

total:           356
image-dependent: 80
text-answerable: 276


In [11]:
from collections import Counter
print(Counter(q["section"] for q in text_qs))

Counter({'General Knowledge RUH': 73, 'Intersections RUH': 44, 'Traffic Lights / Lanes RUH': 32, 'Negligent Driving RUH': 26, 'Pedestrians RUH': 25, 'Fatigue and Defensive Driving RUH': 22, 'Alcohol and Drugs RUH': 18, 'Seat Belts / Restraints RUH': 12, 'Speed Limits RUH': 9, 'ICAC RUH': 3, 'Bicycle Safety RUH': 3, 'Alcohol and Drugs': 2, 'Traffic Lights / Lanes': 2, 'Fatigue and Defensive Driving RUH8': 1, 'Fatigue and Defensive Driving': 1, 'Intersections': 1, 'Negligent Driving': 1, 'Pedestrians RUH4': 1})


In [12]:
def norm(s):
    s = re.sub(r"\s*RUH\d*\s*$", "", s).strip()
    return s

for q in text_qs:
    q["category"] = norm(q["section"])

print(Counter(q["category"] for q in text_qs))

Counter({'General Knowledge': 73, 'Intersections': 45, 'Traffic Lights / Lanes': 34, 'Negligent Driving': 27, 'Pedestrians': 26, 'Fatigue and Defensive Driving': 24, 'Alcohol and Drugs': 20, 'Seat Belts / Restraints': 12, 'Speed Limits': 9, 'ICAC': 3, 'Bicycle Safety': 3})


In [13]:
import random
random.seed(42)

from collections import defaultdict
by_cat = defaultdict(list)
for q in text_qs:
    by_cat[q["category"]].append(q)

# take proportionally, min 1 per category, target 20
target = 20
picked = []
cats = sorted(by_cat, key=lambda c: -len(by_cat[c]))

for c in cats:
    picked.append(random.choice(by_cat[c]))          # 1 each = 11

remaining = target - len(picked)
pool = [q for q in text_qs if q not in picked]
picked += random.sample(pool, remaining)

print(len(picked))
print(Counter(q["category"] for q in picked))

20
Counter({'General Knowledge': 5, 'Fatigue and Defensive Driving': 3, 'Intersections': 2, 'Traffic Lights / Lanes': 2, 'Pedestrians': 2, 'Negligent Driving': 1, 'Alcohol and Drugs': 1, 'Seat Belts / Restraints': 1, 'Speed Limits': 1, 'ICAC': 1, 'Bicycle Safety': 1})


In [14]:
for i, q in enumerate(picked, 1):
    print(f"{i}. [{q['dkt_id']}] {q['category']}")
    print(f"   Q: {q['question']}")
    print(f"   A: {q['answer']}")
    print()

1. [CG034] General Knowledge
   Q: Before driving on a freeway, which of the following should you do?
   A: Make sure your vehicle has enough fuel, oil, water and the correct tyre pressure.

2. [IN005] Intersections
   Q: If a STOP or GIVE WAY sign has been knocked down, for example, as the result of an accident, does the line marked across the road have any meaning?
   A: Yes, it has the same meaning as the sign itself.

3. [LD040] Traffic Lights / Lanes
   Q: Overtaking another vehicle requires good concentration and judgement. Which of the following statements is correct?
   A: You need to watch out for all large, long vehicles which are turning.

4. [ND012] Negligent Driving
   Q: When being overtaken by another vehicle what should you do to help?
   A: Keep left and allow plenty of room to let the other vehicle back in.

5. [PD009] Pedestrians
   Q: When driving near children playing or walking near the edge of the road, you should -
   A: Slow down, and be ready to make a safe st

In [15]:
IMG2 = re.compile(r"\bthis sign\b|\bthis diagram\b|\bshown\b|\bpicture\b|\bimage\b|\billustrat|\bthis symbol\b|\bmirror [A-C]\b|\bvehicle [A-C]\b|\bcar [A-C]\b|\bwhich .{0,20}\bbest\b|\babove\b|\bbelow\b", re.I)

def is_img(q):
    return IMG2.search(q["question"]) or IMG2.search(" ".join(q["options"]))

image_qs2 = [q for q in text_qs if is_img(q)]
text_qs2  = [q for q in text_qs if not is_img(q)]

print(f"newly caught: {len(image_qs2)}")
print(f"text-answerable now: {len(text_qs2)}")

newly caught: 16
text-answerable now: 260


In [16]:
picked = [q for q in picked if q["dkt_id"] != "CG087"]
pool = [q for q in text_qs2 if q not in picked and q["category"] == "General Knowledge"]
picked.append(random.choice(pool))

print(len(picked))
for q in picked[-1:]:
    print(q["dkt_id"], q["question"], "|", q["answer"])

20
CG112 You are driving your vehicle along a street and want to stop for a short time. Are you allowed to double park your vehicle (that is stand it on the road alongside a parked car)? | No, never.


In [17]:
import json, os

golden = []
for i, q in enumerate(sorted(picked, key=lambda x: x["dkt_id"]), 1):
    golden.append({
        "id": f"k{i:02d}",
        "dkt_id": q["dkt_id"],
        "question": q["question"],
        "answer": q["answer"],
        "type": "known",
        "in_kb": True,
        "category": q["category"],
    })

os.makedirs("data/golden", exist_ok=True)
with open("data/golden/golden_set.jsonl", "w", encoding="utf-8") as f:
    for r in golden:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"wrote {len(golden)} known questions")
print(golden[0])

wrote 20 known questions
{'id': 'k01', 'dkt_id': 'AD013', 'question': 'If you take medicine and then drink alcohol -', 'answer': 'It can have a particularly bad effect on your driving ability.', 'type': 'known', 'in_kb': True, 'category': 'Alcohol and Drugs'}


In [18]:
for g in golden:
    print(f"{g['id']} [{g['category']}]")
    print(f"  Q: {g['question']}")
    print(f"  A: {g['answer']}\n")

k01 [Alcohol and Drugs]
  Q: If you take medicine and then drink alcohol -
  A: It can have a particularly bad effect on your driving ability.

k02 [Bicycle Safety]
  Q: In order to provide the minimum safe passing distance for bicycle riders, are you permitted to cross single and double continuous lines?
  A: no.

k03 [General Knowledge]
  Q: When driving at sunset or dawn on a dark day, what should you do?
  A: Turn on your lights on low beam.

k04 [General Knowledge]
  Q: Is it an offence to obstruct clear vision of your number plates?
  A: Yes, at any time.

k05 [General Knowledge]
  Q: Before driving on a freeway, which of the following should you do?
  A: Make sure your vehicle has enough fuel, oil, water and the correct tyre pressure.

k06 [General Knowledge]
  Q: What must you do if you miss your exit on a freeway?
  A: Continue until you reach the next appropriate exit.

k07 [General Knowledge]
  Q: You are driving your vehicle along a street and want to stop for a short time.

In [19]:
inferred_raw = [
    ("k01", "Is it bad to drink if I'm on medication?"),
    ("k02", "..."),
    # 10 total
]